# 25. Canonicalization Test (Pipeline 06)

- Docs: `docs_eng/pipeline/06_canonicalization.md` / `docs/pipeline/06_canonicalization.md`
- Prerequisite: 20-24 stage checks should already pass.
- Input: normalized preprocessed pose dataframe from the same previous-stage setup used by 24.
- Output: additive `canon` candidate columns and `canonicalization_report`; downstream coordinate mode remains `norm`.
- Checks: previous-stage setup, direct canonicalization output/provenance, candidate summary and prior evidence, compact visual comparison, diagnostics, and pipeline integration.
- Policy: candidate evidence only; no score gravity or final-score contribution is emitted here.


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import warnings

import pandas as pd
import plotly.graph_objects as go

from movement.canonicalization import (
    CanonicalizationConfig,
    MovementPlaneAlignmentConfig,
    ProtocolHeightLateralWidthAlignmentConfig,
    apply_canonicalization,
)
from movement.config import CONNECTIONS, LANDMARKS
from movement.floor_reference import FloorReferenceConfig
from movement.normalization import check_normalization_result
from movement.pipeline import (
    NormalizationConfig,
    PipelineConfig,
    PreprocessingConfig,
    ValidationConfig,
    run_pipeline,
)
from movement.stage_context import prepare_previous_stage_inputs
from movement.visualization import create_pose_comparison_animation

print('imports OK')


## Data Setup

Prepare the same previous-stage input chain used by the prior stage checks: validation, annotation, exercise-definition loading, preprocessing, and normalization. Canonicalization is tested on `norm_df`, not on raw pose data.


In [ ]:
pose_csv = 'data/pose/mediapipe/no_consent/20260517/p01_squat_set1_output_pose.csv'
annotation_csv = 'data/pose/mediapipe/no_consent/20260517/p01_squat_set1_annotation.csv'
TARGET_EXERCISE_ID = 'draft_squat'


def estimate_frame_duration_ms(dataframe, default_ms=33):
    if 'timestamp' not in dataframe.columns:
        return default_ms
    dt = dataframe['timestamp'].astype(float).diff().dropna()
    if dt.empty:
        return default_ms
    median_dt = float(dt.median())
    if median_dt <= 0:
        return default_ms
    return max(1, int(round(median_dt * 1000)))


pre_config = PreprocessingConfig(enabled=True)
norm_config = NormalizationConfig(
    enabled=True,
    keep_reference_columns=True,
    model_depth_scale=1.0,
)
stage_inputs = prepare_previous_stage_inputs(
    prepare_until='normalization',
    pose_csv=pose_csv,
    annotation_csv=annotation_csv,
    exercise_id=TARGET_EXERCISE_ID,
    landmarks=LANDMARKS,
    preprocessing_config=pre_config,
    normalization_config=norm_config,
)

df_raw = stage_inputs.raw_df
df_annotated = stage_inputs.annotated_df
val_report = stage_inputs.validation_report
ann_report = stage_inputs.annotation_report
exercise_def = stage_inputs.exercise_definition
pre_df = stage_inputs.preprocessed_df
pre_report = stage_inputs.preprocessing_report
norm_df = stage_inputs.normalized_df
norm_report = stage_inputs.normalization_report
TARGET_DEFINITIONS_DIR = stage_inputs.definitions_dir

check_report = check_normalization_result(norm_df)
assert check_report['passed'] is True
frame_duration_ms = estimate_frame_duration_ms(norm_df)

setup_summary = pd.DataFrame([
    {'item': 'frames_loaded', 'value': len(df_raw)},
    {'item': 'validation_passed', 'value': val_report['passed']},
    {'item': 'structural_validation_passed', 'value': val_report.get('structural_passed')},
    {'item': 'analysis_frames', 'value': f"{ann_report['num_analysis_frames']} / {ann_report['num_total_frames']}"},
    {'item': 'exercise_id', 'value': exercise_def.exercise_id},
    {'item': 'movement_template_id', 'value': exercise_def.classification['movement_template_id']},
    {'item': 'preprocessing_invalid_frames', 'value': pre_report['num_invalid_frames']},
    {'item': 'normalization_scale_value', 'value': round(float(norm_report['scale_value']), 6)},
    {'item': 'model_depth_scale', 'value': norm_report['model_depth_scale']},
    {'item': 'preprocessed_shape', 'value': pre_df.shape},
    {'item': 'normalized_shape', 'value': norm_df.shape},
    {'item': 'definitions_dir', 'value': str(TARGET_DEFINITIONS_DIR)},
])
display(setup_summary)

if val_report.get('warnings'):
    display(pd.DataFrame(val_report['warnings']))
if not val_report['passed']:
    print('NOTE: structural validation failed; inspect before canonicalization.')


## Direct Canonicalization Test

Create canonical candidate evidence from the normalized preprocessed dataframe. `report_only=True` keeps downstream stages on `norm` coordinates.


In [ ]:
support_config = FloorReferenceConfig(
    enabled=True,
    method='support_contact_plane',
    coordinate_mode='norm',
    vertical_axis='y',
    support_landmarks=[
        'left_heel',
        'right_heel',
        'left_foot_index',
        'right_foot_index',
    ],
    diagnostic_landmarks=[
        'left_heel',
        'right_heel',
        'left_foot_index',
        'right_foot_index',
    ],
    visibility_threshold=0.7,
    max_anchor_residual_torso=0.08,
    correction_transform='rigid_rotation',
    camera_pitch_deg=0.0,
    camera_roll_deg=0.0,
    correction_strength=1.0,
    max_correction_torso=0.25,
)

movement_config = MovementPlaneAlignmentConfig(
    enabled=True,
    method='principal_motion_plane',
    fit_landmarks=[
        'left_hip',
        'left_knee',
        'left_ankle',
        'right_hip',
        'right_knee',
        'right_ankle',
    ],
    minimum_visible_landmark_ratio=0.7,
    correction_strength=0.5,
    max_rotation_deg=20.0,
    preserve_out_of_plane_residual=True,
)

protocol_height_config = ProtocolHeightLateralWidthAlignmentConfig(
    enabled=True,
    observed_height_level='H2',
    recommended_height_level='H2',
    require_height_match=True,
    correction_strength=0.3,
    max_scale_change=0.20,
    max_correction_torso=0.15,
    min_depth_offset_torso=0.05,
    visibility_threshold=0.6,
)

canonical_config = CanonicalizationConfig(
    enabled=True,
    coordinate_mode='norm',
    output_prefix='canon',
    report_only=True,
    downstream_coordinate_mode='norm',
    support_plane_alignment=support_config,
    movement_plane_alignment=movement_config,
    protocol_height_lateral_width_alignment=protocol_height_config,
)

canon_df, canon_report = apply_canonicalization(
    df=norm_df,
    landmarks=LANDMARKS,
    config=canonical_config,
)

active_canonicalization_config = pd.DataFrame([
    {'area': 'canonicalization', 'setting': 'enabled', 'value': canonical_config.enabled},
    {'area': 'canonicalization', 'setting': 'coordinate_mode', 'value': canonical_config.coordinate_mode},
    {'area': 'canonicalization', 'setting': 'output_prefix', 'value': canonical_config.output_prefix},
    {'area': 'canonicalization', 'setting': 'report_only', 'value': canonical_config.report_only},
    {'area': 'canonicalization', 'setting': 'downstream_coordinate_mode', 'value': canonical_config.downstream_coordinate_mode},
    {'area': 'support_plane_alignment', 'setting': 'enabled', 'value': support_config.enabled},
    {'area': 'movement_plane_alignment', 'setting': 'enabled', 'value': movement_config.enabled},
    {'area': 'protocol_height_lateral_width_alignment', 'setting': 'enabled', 'value': protocol_height_config.enabled},
])
display(active_canonicalization_config)

print('canonicalization status:', canon_report['status'])
print('candidate:', canon_report['candidate_available'], canon_report['candidate_confidence'], canon_report['burden_level'])


## Check 1: Output Columns and Provenance

Confirm that raw, `norm`, and `canon` families are additive and that preprocessing reliability/usability metadata survives canonicalization.


In [ ]:
assert canonical_config.report_only is True
assert canonical_config.downstream_coordinate_mode == 'norm'
assert isinstance(canon_report['candidate_available'], bool)
assert canon_report['candidate_confidence'] in {'not_available', 'high', 'moderate', 'low', 'not_emitted'}
assert 'score_gravity' not in canon_report
assert 'final_score' not in canon_report

missing_raw_norm = []
missing_canon = []
for landmark in LANDMARKS:
    for axis in ['x', 'y', 'z']:
        raw_col = f'{landmark}_{axis}'
        norm_col = f'{landmark}_norm_{axis}'
        canon_col = f'{landmark}_canon_{axis}'
        if raw_col not in canon_df.columns:
            missing_raw_norm.append(raw_col)
        if norm_col not in canon_df.columns:
            missing_raw_norm.append(norm_col)
        if canon_col not in canon_df.columns:
            missing_canon.append(canon_col)
assert not missing_raw_norm, f'missing raw/norm columns: {missing_raw_norm[:8]}'
assert not missing_canon, f'missing canon columns: {missing_canon[:8]}'

required_metadata_columns = [
    'preprocessing_valid',
    'canonicalization_valid',
    'canonicalization_candidate_available',
    'canonicalization_candidate_confidence',
    'canonicalization_burden_level',
    'canonicalization_confidence',
    'canonicalization_correction_abs_frame',
]
missing_metadata_columns = [col for col in required_metadata_columns if col not in canon_df.columns]
assert not missing_metadata_columns, f'missing metadata columns: {missing_metadata_columns}'

usable_columns = [col for col in canon_df.columns if col.endswith('_usable')]
source_columns = [col for col in canon_df.columns if col.endswith('_preprocessing_source')]
corrected_columns = [col for col in canon_df.columns if '_corrected_3d_hypothesis_' in col]
assert not corrected_columns, 'corrected-3D-hypothesis columns should not be emitted by this 25 check'

contract_summary = pd.DataFrame([
    {'item': 'raw_norm_coordinate_families_present', 'value': True},
    {'item': 'canon_coordinate_columns_present', 'value': len(LANDMARKS) * 3},
    {'item': 'preprocessing_valid_preserved', 'value': 'preprocessing_valid' in canon_df.columns},
    {'item': 'landmark_usable_columns_preserved', 'value': len(usable_columns)},
    {'item': 'landmark_source_columns_preserved', 'value': len(source_columns)},
    {'item': 'score_gravity_emitted', 'value': 'score_gravity' in canon_report},
    {'item': 'corrected_3d_hypothesis_columns_emitted', 'value': len(corrected_columns)},
])
display(contract_summary)
print('PASS: canonicalization output contract and previous-stage provenance are valid')


## Check 2: Candidate Summary and Prior Evidence

Review the whole-candidate confidence/burden surface separately from prior-level evidence. Prior rows expose status and a key diagnostic, not a separate score.


In [ ]:
prior_reports = canon_report.get('prior_reports', {})
support_report = prior_reports.get('support_plane_alignment')
movement_report = prior_reports.get('movement_plane_alignment')
protocol_height_report = prior_reports.get('protocol_height_lateral_width_alignment')


def prior_available(prior_report):
    return bool(prior_report and prior_report.get('status') in {'applied', 'warning'})


def prior_reason(prior_report):
    if not prior_report:
        return 'not_configured'
    notes = prior_report.get('confidence_notes', []) or []
    if notes:
        return notes[0]
    if prior_available(prior_report):
        return 'available'
    return str(prior_report.get('status', 'not_available'))


def support_key_metric(prior_report):
    if not prior_report:
        return ''
    residual = (prior_report.get('anchor_residual_summary') or {}).get('max')
    return f"anchor_frames={prior_report.get('num_anchor_frames')}; residual_max={residual}"


def movement_key_metric(prior_report):
    if not prior_report:
        return ''
    residual = (prior_report.get('out_of_plane_residual_ratio_after') or {}).get('p90')
    return f"rotation_deg={prior_report.get('applied_rotation_deg')}; residual_p90={residual}"


def protocol_key_metric(prior_report):
    if not prior_report:
        return ''
    return (
        f"height={prior_report.get('observed_height_level')}->"
        f"{prior_report.get('recommended_height_level')}; "
        f"max_scale_delta={prior_report.get('max_scale_delta')}"
    )


candidate_summary = pd.DataFrame([
    {
        'candidate_available': canon_report['candidate_available'],
        'candidate_confidence': canon_report['candidate_confidence'],
        'burden_level': canon_report['burden_level'],
        'data_confidence': canon_report['data_confidence']['level'],
        'confidence_reasons': ', '.join(canon_report['data_confidence']['reasons']) or 'none',
        'downstream_coordinate_mode': canon_report['downstream_coordinate_mode'],
        'max_correction_torso': canon_report['max_correction_torso'],
        'median_correction_torso': canon_report['median_correction_torso'],
        'residual_after_fit_torso': canon_report['residual_after_fit_torso'],
    }
])

prior_evidence = pd.DataFrame([
    {
        'prior_id': 'support_plane_alignment',
        'configured_on': support_config.enabled,
        'candidate_available': prior_available(support_report),
        'reason': prior_reason(support_report),
        'key_metric': support_key_metric(support_report),
    },
    {
        'prior_id': 'movement_plane_alignment',
        'configured_on': movement_config.enabled,
        'candidate_available': prior_available(movement_report),
        'reason': prior_reason(movement_report),
        'key_metric': movement_key_metric(movement_report),
    },
    {
        'prior_id': 'protocol_height_lateral_width_alignment',
        'configured_on': protocol_height_config.enabled,
        'candidate_available': prior_available(protocol_height_report),
        'reason': prior_reason(protocol_height_report),
        'key_metric': protocol_key_metric(protocol_height_report),
    },
])

display(candidate_summary)
display(prior_evidence)

assert canon_report['downstream_coordinate_mode'] == 'norm'
assert set(prior_evidence['prior_id']) == {
    'support_plane_alignment',
    'movement_plane_alignment',
    'protocol_height_lateral_width_alignment',
}
print('PASS: candidate summary and prior evidence are separated')


## Check 3: Visual Comparison

Compare the base `norm` family with the candidate `canon` family using the recording-view camera. This visual check is for candidate-evidence inspection only.


In [ ]:
recording_view_camera = dict(
    eye=dict(x=0.0, y=-2.5, z=0.0),
    center=dict(x=0.0, y=0.0, z=0.0),
    up=dict(x=0.0, y=0.0, z=1.0),
    projection=dict(type='orthographic'),
)


def apply_recording_view_camera(fig):
    fig.update_layout(scene_camera=recording_view_camera)
    return fig


fig_compare = create_pose_comparison_animation(
    df=canon_df,
    landmarks=LANDMARKS,
    connections=CONNECTIONS,
    coord_modes=('norm', 'canon'),
    names=('Normalized', 'Canonical candidate'),
    frame_duration=frame_duration_ms,
    height=750,
    width=1000,
    show_text=False,
    title='Normalized vs Canonical Candidate Coordinates',
)

apply_recording_view_camera(fig_compare)
fig_compare.show()


## Check 4: Diagnostics

Inspect support-plane residuals and correction magnitude. These are provenance and data-confidence signals, not movement-quality deductions.


In [ ]:
support_residual_max = None
if support_report:
    support_residual_max = (support_report.get('anchor_residual_summary') or {}).get('max')
movement_residual_p90 = None
if movement_report:
    movement_residual_p90 = (movement_report.get('out_of_plane_residual_ratio_after') or {}).get('p90')

diagnostic_table = pd.DataFrame([
    {'item': 'support_anchor_frames', 'value': None if not support_report else support_report.get('num_anchor_frames')},
    {'item': 'support_anchor_residual_max', 'value': support_residual_max},
    {'item': 'movement_rotation_deg', 'value': None if not movement_report else movement_report.get('applied_rotation_deg')},
    {'item': 'movement_residual_p90_after', 'value': movement_residual_p90},
    {'item': 'protocol_height_match', 'value': None if not protocol_height_report else protocol_height_report.get('height_match')},
    {'item': 'protocol_height_max_scale_delta', 'value': None if not protocol_height_report else protocol_height_report.get('max_scale_delta')},
    {'item': 'max_correction_torso', 'value': canon_report['max_correction_torso']},
    {'item': 'median_correction_torso', 'value': canon_report['median_correction_torso']},
    {'item': 'data_confidence', 'value': canon_report['data_confidence']['level']},
])
display(diagnostic_table)

fig_diag = go.Figure()
for landmark in support_config.diagnostic_landmarks:
    col = f'{landmark}_canon_support_plane_height'
    if col in canon_df.columns:
        fig_diag.add_trace(
            go.Scatter(
                x=canon_df['frame'],
                y=canon_df[col],
                mode='lines',
                name=col,
            )
        )

if 'canonicalization_correction_abs_frame' in canon_df.columns:
    fig_diag.add_trace(
        go.Scatter(
            x=canon_df['frame'],
            y=canon_df['canonicalization_correction_abs_frame'],
            mode='lines',
            name='canonicalization_correction_abs_frame',
            line=dict(dash='dash'),
        )
    )

if 'canonicalization_lateral_width_scale_delta_frame' in canon_df.columns:
    fig_diag.add_trace(
        go.Scatter(
            x=canon_df['frame'],
            y=canon_df['canonicalization_lateral_width_scale_delta_frame'],
            mode='lines',
            name='lateral_width_scale_delta_frame',
            line=dict(dash='dot'),
        )
    )

fig_diag.update_layout(
    title='Canonicalization Diagnostics',
    xaxis_title='Frame',
    yaxis_title='torso_length_ratio',
    height=420,
    width=950,
)
fig_diag.show()


## Check 5: Pipeline Integration

Run the pipeline with canonicalization enabled and confirm that the report is emitted while downstream coordinate mode remains `norm`.


In [ ]:
pipe_config = PipelineConfig()
pipe_config.validation = ValidationConfig(enabled=True)
pipe_config.preprocessing = PreprocessingConfig(enabled=True)
pipe_config.normalization = norm_config
pipe_config.canonicalization = canonical_config
pipe_config.exercise_definition.exercise_id = TARGET_EXERCISE_ID
pipe_config.exercise_definition.definitions_dir = str(TARGET_DEFINITIONS_DIR)

with warnings.catch_warnings(record=True):
    warnings.simplefilter('always')
    pipe_df, pipe_report = run_pipeline(
        df_annotated,
        config=pipe_config,
        landmarks=LANDMARKS,
    )

assert 'preprocessing' in pipe_report
assert 'normalization' in pipe_report
assert 'canonicalization' in pipe_report
assert pipe_report['exercise_definition']['exercise_id'] == TARGET_EXERCISE_ID
assert pipe_report['canonicalization']['report_only'] is True
assert pipe_report['canonicalization']['downstream_coordinate_mode'] == 'norm'
assert isinstance(pipe_report['canonicalization']['candidate_available'], bool)
assert 'preprocessing_valid' in pipe_df.columns
assert any(col.endswith('_canon_x') for col in pipe_df.columns)
assert 'score_gravity' not in pipe_report['canonicalization']

pipeline_summary = pd.DataFrame([
    {'item': 'steps_executed', 'value': list(pipe_report.keys())},
    {'item': 'pipeline_exercise_id', 'value': pipe_report['exercise_definition']['exercise_id']},
    {'item': 'pipeline_output_shape', 'value': pipe_df.shape},
    {'item': 'pipeline_candidate_available', 'value': pipe_report['canonicalization']['candidate_available']},
    {'item': 'pipeline_candidate_confidence', 'value': pipe_report['canonicalization']['candidate_confidence']},
    {'item': 'pipeline_burden_level', 'value': pipe_report['canonicalization']['burden_level']},
    {'item': 'pipeline_downstream_coordinate_mode', 'value': pipe_report['canonicalization']['downstream_coordinate_mode']},
])
display(pipeline_summary)
print('PASS: canonicalization step is integrated without changing downstream coordinate mode')


## Check Summary

Summarize whether canonicalization is available as candidate evidence and whether it remains separated from scoring.


In [ ]:
check_summary = pd.DataFrame([
    {'check': 'previous_stage_chain_prepared', 'passed': True, 'note': 'validation -> annotation -> exercise definition -> preprocessing -> normalization'},
    {'check': 'candidate_available', 'passed': bool(canon_report['candidate_available']), 'note': canon_report['candidate_confidence']},
    {'check': 'burden_reported', 'passed': canon_report['burden_level'] in {'none', 'low', 'moderate', 'high'}, 'note': canon_report['burden_level']},
    {'check': 'prior_evidence_table_present', 'passed': len(prior_evidence) == 3, 'note': ', '.join(prior_evidence['prior_id'])},
    {'check': 'downstream_mode_norm', 'passed': canon_report['downstream_coordinate_mode'] == 'norm', 'note': canon_report['downstream_coordinate_mode']},
    {'check': 'score_gravity_absent', 'passed': 'score_gravity' not in canon_report, 'note': 'scoring policy owns score gravity'},
    {'check': 'pipeline_integration', 'passed': 'canonicalization' in pipe_report, 'note': pipe_report['canonicalization']['candidate_confidence']},
])
display(check_summary)
assert check_summary['passed'].all()
print('PASS: 25 canonicalization stage check completed')
